---
# Plan (Atumate the process)
1. Import the labels files (use the sql.config)
2. Join the labels with the clustring results
3. perform analysis (next section)
---
* Generate clumpiness figure
* find k-mer whithin each cluster that singnifies properties -> 
----
To do:
1. Join multiple labels togather, per config load.
2. Join nodes and clusters tables with the labels.
3. Find out node spcific k-kmer via the clusters tree (json).
4. Use clumpiness to visualize and / or pinpoint clusters -> get a list of clusters.

In [ ]:
# Import packages
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import os

In [ ]:
# Defining imports paths
subject_id = 7
tms_output_path = os.path.join("tms_output",f"covid_vaccine_new-subject{subject_id}")

In [ ]:
# Loading clusters data into python
node_info = pd.read_csv(os.path.join(tms_output_path, "node_info.csv"), index_col=0).reset_index()
clusters = pd.read_csv(os.path.join(tms_output_path, "clusters.csv"), index_col=0)

In [ ]:
path = "tms_input\\covid_vaccine_new-subject7"
labels_abtarget = pd.read_csv(os.path.join(path, "labels-ab_target.csv"))
labels_timepoint = pd.read_csv(os.path.join(path, "labels-time_point.csv"))

In [ ]:
demo = pd.concat([labels_abtarget, labels_timepoint], axis=1)
demo["concated"] = labels_abtarget.label + "." + labels_timepoint.label

In [ ]:
demo.iloc[:,[-3,-1]].set_index("item").rename({"concated":"label"}, axis=1).to_csv("demo_label.csv")

In [ ]:
final = demo.iloc[:,-2:].set_index("label")

In [ ]:
node_info[node_info.node == 42972]

In [ ]:
if False:
    cells_clusters = pd.read_csv("tms_input\\covid_vaccine_new-subject7\\labels-ab_target.csv", index_col=0)
    labels_1 = pd.read_csv("tms_input\\covid_vaccine_new-subject7\\labels-ab_target.csv", index_col=0)
    labels_2 = pd.read_csv('tms_input\\covid_vaccine_new-subject7\\labels-time_point.csv', index_col=0)

    from source.toomanycells.toomanycells import TooManyCells as tmc

    tmc_indir = "tms_input\\covid_vaccine_new-subject7"
    tmc_outdir = "tms_output\\covid_vaccine_new-subject7"
    tmc_object = tmc(tmc_indir,
                    tmc_outdir,
                    input_is_matrix_market=True)

    tmc_object.update_cell_annotations(df=pd.read_csv("tms_input\\covid_vaccine_new-subject7\\labels-ab_target.csv", index_col=0),
                                    column="cell_annotations")

    tmc_object.run_spectral_clustering()

    tmc_object.store_outputs()

-----
Dev code

In [40]:
from source.helpers import read_json
import pandas as pd
import numpy as np
import os
import json
import pandas as pd
import sys


# 
class cluster_analysis():
    ####
    def __init__(self,
                 subject_id : int):
        
        """
        subject_id : int -> number of the subject on which the analysis will be performed.
        """

        # file paths
        self.subject = str(subject_id)
        self.config = read_json()
        self.path_tmc_input = os.path.join("tms_input", f"{self.config["database"]["db_name"]}-subject{self.subject}")
        self.path_tmc_output = os.path.join("tms_output", f"{self.config["database"]["db_name"]}-subject{self.subject}")

        # Metadata labels dict
        labels = self.config["database"]["metadata_labels"].split(",")
        relabels = self.config["database"]["metadata_relabels"].split(",")
        if (relabels != "") & (len(relabels) == len(labels)):
            self.labels_dict = {i:j for i,j in zip(labels, relabels)}
        else:
            self.labels_dict = {i:j for i,j in zip(labels, labels)}

        # Required files
        tmc_files = ["cells_clusters_info.csv", "cluster_list.json", "cluster_tree.json", "clusters.csv", "graph.json", "node_info.csv"]
        self.req_files = {}

        # TMC output files -> paths into required dict
        for file in tmc_files:
            file_name = file.split(".")[0]
            self.req_files[file_name] = os.path.join(self.path_tmc_output, file)

        # labels files -> paths into required dict
        for label in self.labels_dict:
            self.req_files[label] = os.path.join(self.path_tmc_input, f"labels-{self.labels_dict[label]}.csv")

        # Verify that all of the required files indeed exsits:
        bool_files = [os.path.exists(i) for i in list(self.req_files.values())]

        if all(bool_files) is False:
            print(f"Missing files: {[i for i,j in zip(list(self.req_files.values()), bool_files) if j is False]}")

        # Joining metadata to cells-clusters fule:
        self.cell_clusters = pd.read_csv(self.req_files["cells_clusters_info"], index_col=0)

        for label in list(self.labels_dict.keys()):
            temp_lfile = pd.read_csv(self.req_files[label], index_col=0)
            temp_lfile.columns = [self.labels_dict[label]]
            self.cell_clusters = pd.merge(left=self.cell_clusters, right=temp_lfile, left_index=True, right_index=True, how="inner")


    ###
    def get_clusters(self,
                     save_output : bool = True):
        return self.cell_clusters


    ####
    def extract_nodes_dfs(self,
                          tree_json_path : str, 
                          output_path : str = None,
                          save_output : bool = True):
        
        """
        json_path : str -> path of the 
        output_path : str ->
        save_output : bool ->
        """
        # Increase recursion depth for very large trees
        sys.setrecursionlimit(200000)

        # file output:
        if output_path is None:
            output_path = os.path.join(self.path_tmc_output, "nodes_cells.csv")

        with open(tree_json_path, 'r') as f:
            tree = json.load(f)

        node_list = []
        node_id_counter = 0

        def traverse(node):
            nonlocal node_id_counter
            current_id = node_id_counter
            node_id_counter += 1
            
            meta, children = node
            
            # 1. Collect barcodes from this node and all descendants
            # (This uses a small internal helper to stay efficient)
            def get_all_barcodes(n):
                m, c = n
                bc = [item['_barcode']['unCell'] for item in m['_item']] if m.get('_item') else []
                for child in c:
                    bc.extend(get_all_barcodes(child))
                return bc

            barcodes = get_all_barcodes(node)
            
            # 2. Store node data
            node_list.append({
                'node_id': current_id,
                'cell_count': len(barcodes),
                'barcodes': ",".join(barcodes)
                              })

            # 3. Recursively visit children (DFS order)
            for child in children:
                traverse(child)

        traverse(tree)
        
        self.node_cells = pd.DataFrame(node_list)

        if save_output:
            self.node_cells.to_csv(output_path, index=False)

        return self.node_cells

In [ ]:
tree_path = 'tms_output\\covid_vaccine_new-subject7\\cluster_tree.json', 'node_to_barcodes_dfs.csv'
cluster_test = cluster_analysis(subject_id=7)
cell_clusters = cluster_test.cell_clusters
cell_clusters.head(5)

,sp_cluster,sp_path,ab_target,time_point
LYLQMNSLRAEDTAVYYCAR,25523,25523/25522/25521/25509/25508/25507/25506/2550...,Non-Spike B,1 baseline
YLQMNSLRAEDTAVYYCARD,25473,25473/25472/25471/25469/25468/25467/25455/2542...,Non-Spike B,1 baseline
sLKLSsVTAADTAVYYCARG,29669,29669/29668/29667/29661/29660/29648/29647/2951...,Non-Spike B,1 baseline
NTLYLQMNSLRAEDTAVYYC,25513,25513/25512/25511/25510/25509/25508/25507/2550...,Non-Spike B,1 baseline
TLYLQMNSLRAEDTAVYYCA,25516,25516/25515/25511/25510/25509/25508/25507/2550...,Non-Spike B,1 baseline


taks:
1. add another folder to the program scheme -> "tmc_analysis"
2. change folders names from "tms" to "tmc"
3. create "tmc analysis" script in source
4. create tutorial notebook
5. look into clumpintess